In [3]:
# ============================================
# 🧠 ADVANCED MULTI-AGENT SYSTEM
# (Async + Shared Memory + Groq)
# ============================================

!pip install groq nest_asyncio

import asyncio
import nest_asyncio
from groq import Groq
from google.colab import userdata

nest_asyncio.apply()

# ============================================
# 🔑 LOAD API KEY
# ============================================

api_key = userdata.get("groq_api_key")

if api_key is None:
    raise ValueError("❌ Add GROQ_API_KEY in Colab Secrets")

client = Groq(api_key=api_key)


# ============================================
# 🧠 SHARED MEMORY (ADVANCED)
# ============================================

class SharedMemory:
    def __init__(self):
        self.data = {}
        self.logs = []

    def update(self, agent, value):
        self.data[agent] = value
        self.logs.append(f"{agent} updated memory")

    def get(self, agent):
        return self.data.get(agent, "")

    def get_all(self):
        return self.data

    def show_logs(self):
        print("\n📝 MEMORY LOGS:")
        for log in self.logs:
            print("•", log)


# ============================================
# 🔹 LLM FUNCTION (ASYNC)
# ============================================

async def ask_llm(prompt):
    loop = asyncio.get_event_loop()

    response = await loop.run_in_executor(
        None,
        lambda: client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "You are an expert AI research assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.6,
            max_tokens=400
        )
    )

    return response.choices[0].message.content


# ============================================
# 🤖 AGENTS
# ============================================

# 🔍 Search Agent
async def search_agent(goal, memory):
    print("\n🔍 Search Agent running...")

    prompt = f"""
    Goal: {goal}

    Find:
    - Market trends
    - Statistics
    - Key insights
    """

    result = await ask_llm(prompt)
    memory.update("search", result)


# 📊 Analyst Agent
async def analyst_agent(goal, memory):
    print("\n📊 Analyst Agent running...")

    data = memory.get("search")

    prompt = f"""
    Goal: {goal}

    Analyze:
    {data}

    Provide deep insights and patterns.
    """

    result = await ask_llm(prompt)
    memory.update("analyst", result)


# ✍️ Writer Agent
async def writer_agent(goal, memory):
    print("\n✍️ Writer Agent running...")

    analysis = memory.get("analyst")

    prompt = f"""
    Write a professional report using:
    {analysis}

    Include headings and structured format.
    """

    result = await ask_llm(prompt)
    memory.update("writer", result)


# ✅ QA Agent
async def qa_agent(goal, memory):
    print("\n✅ QA Agent running...")

    report = memory.get("writer")

    prompt = f"""
    Review and improve this report:
    {report}

    Fix clarity, grammar, and quality.
    """

    result = await ask_llm(prompt)
    memory.update("qa", result)


# 🧑‍⚖️ Decision Agent
async def decision_agent(goal, memory):
    print("\n🧑‍⚖️ Decision Agent running...")

    all_data = memory.get_all()

    prompt = f"""
    Goal: {goal}

    Combine all outputs:
    {all_data}

    Provide:
    - Executive Summary
    - Key Insights
    - Opportunities
    - Risks
    - Final Recommendation
    """

    result = await ask_llm(prompt)
    memory.update("final", result)


# ============================================
# 🎯 ORCHESTRATOR
# ============================================

async def run_system(goal):
    print("🎯 Goal:", goal)

    memory = SharedMemory()

    # Sequential pipeline (can be extended to parallel)
    await search_agent(goal, memory)
    await analyst_agent(goal, memory)
    await writer_agent(goal, memory)
    await qa_agent(goal, memory)
    await decision_agent(goal, memory)

    memory.show_logs()

    return memory.get("final")


# ============================================
# ▶️ RUN
# ============================================

goal = "Write a comprehensive market report on EV industry trends in India for 2025"

result = asyncio.run(run_system(goal))

print("\n✅ FINAL OUTPUT:\n")
print(result)

🎯 Goal: Write a comprehensive market report on EV industry trends in India for 2025

🔍 Search Agent running...

📊 Analyst Agent running...

✍️ Writer Agent running...

✅ QA Agent running...

🧑‍⚖️ Decision Agent running...

📝 MEMORY LOGS:
• search updated memory
• analyst updated memory
• writer updated memory
• qa updated memory
• final updated memory

✅ FINAL OUTPUT:

**Comprehensive Market Report on EV Industry Trends in India for 2025**

**Executive Summary**

The Electric Vehicle (EV) industry in India is witnessing rapid growth, driven by government initiatives, increasing environmental concerns, and technological advancements. This report provides an in-depth analysis of the EV market trends, statistics, and key insights in India for 2025, highlighting the key drivers and challenges that will shape the industry's future.

**Market Trends**

1. **Government Support**: The Indian government has set ambitious targets to promote EV adoption, including a goal of 30% of all new vehicle